# 📚 BioRAG — Notebook 3: PDF Ingestion Pipeline
### Build a Searchable Knowledge Base from Your Own Papers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/genesight-biorag/blob/main/notebooks/03_BioRAG_Ingestion.ipynb)

---

## What this notebook does

**BioRAG** is a Retrieval-Augmented Generation (RAG) system tailored for biomedical literature. You bring your own PDFs — research papers, review articles, protocols — and BioRAG turns them into a searchable knowledge base you can query in natural language.

**Pipeline steps:**
1. Upload your PDF files directly in Colab
2. Extract and clean text from each paper
3. Chunk text into semantically coherent passages
4. Generate dense vector embeddings using a biomedical language model
5. Store embeddings in ChromaDB — a lightweight local vector database
6. Test retrieval with example queries

**Why RAG instead of just asking Claude?**
Claude's training data has a knowledge cutoff and doesn't include your specific papers. RAG grounds the AI's answers in *your* literature, with exact citations back to the source passage. This is the difference between "I think methylation arrays work like this" and "According to Liu et al. 2023, page 4: ..."

---

> **🔒 Privacy note:** Your PDFs are processed entirely within your Colab session and are never stored or transmitted. The notebook stores only embeddings (vector representations), not the raw text, in the persistent database.

## ⚙️ Setup & Installations

We use:
- **`pypdf`** — robust PDF text extraction
- **`sentence-transformers`** — biomedical embedding model (`BiomedBERT`)
- **`chromadb`** — lightweight vector database (runs in-process, no server needed)
- **`anthropic`** — Claude API for the Q&A layer (Notebook 4)

In [ ]:
!pip install pypdf sentence-transformers chromadb anthropic tqdm --quiet
print("✅ Packages installed")

In [ ]:
import os
import re
import json
import hashlib
from pathlib import Path
from typing import List, Dict, Tuple

import chromadb
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm
import numpy as np

# For Colab file upload
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f"✅ Imports complete | Running in Colab: {IN_COLAB}")

# Paths
PDF_DIR = Path("pdfs")
DB_DIR  = Path("biorag_db")
PDF_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)

---
## 📤 Step 1: Upload Your PDFs

Run the cell below to open a file picker. Select all your papers at once (hold Cmd/Ctrl for multi-select). 

**Recommended:** 5–50 papers is ideal. More than ~200 papers may slow down retrieval without chunking optimizations.

In [ ]:
if IN_COLAB:
    print("📂 Select your PDF files...")
    uploaded = files.upload()
    for filename, content in uploaded.items():
        if filename.endswith('.pdf'):
            dest = PDF_DIR / filename
            with open(dest, 'wb') as f:
                f.write(content)
            print(f"   ✅ Saved: {filename} ({len(content)/1e6:.1f} MB)")
else:
    # Local development: place PDFs in ./pdfs/ directory
    print(f"📂 Place your PDFs in: {PDF_DIR.resolve()}")

pdf_files = list(PDF_DIR.glob("*.pdf"))
print(f"\n📚 Total PDFs found: {len(pdf_files)}")
for p in pdf_files:
    size_kb = p.stat().st_size / 1e3
    print(f"   • {p.name} ({size_kb:.0f} KB)")

---
## 📄 Step 2: Extract & Clean Text

PDF text extraction is messier than it looks — headers, footers, figure captions, references sections, and column layouts all create noise. We apply a cleaning pipeline that:

- Removes page numbers, headers/footers
- Collapses hyphenated line-breaks (common in two-column journals)
- Strips reference sections (to avoid the model citing from bibliography)
- Normalizes whitespace

In [ ]:
def clean_pdf_text(raw_text: str) -> str:
    """Clean extracted PDF text for better chunking and retrieval."""
    # Fix hyphenated line breaks (two-column journal artifact)
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', raw_text)

    # Collapse single newlines within paragraphs, preserve double newlines
    text = re.sub(r'(?<!\n)\n(?!\n)', ' ', text)

    # Remove page numbers (standalone digits on a line)
    text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)

    # Remove common journal header artifacts
    text = re.sub(r'(Downloaded from|Copyright|All rights reserved|doi:|DOI:)[^\n]*', '', text)

    # Truncate at References section (avoid retrieval from bibliography)
    ref_match = re.search(
        r'\n\s*(References|Bibliography|Works Cited|Literature Cited)\s*\n',
        text, re.IGNORECASE
    )
    if ref_match:
        text = text[:ref_match.start()]

    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()


def extract_pdf(pdf_path: Path) -> Dict:
    """Extract text and metadata from a PDF."""
    reader = PdfReader(str(pdf_path))
    meta = reader.metadata or {}

    pages_text = []
    for page_num, page in enumerate(reader.pages, 1):
        try:
            pages_text.append(page.extract_text() or "")
        except Exception:
            pages_text.append("")  # Skip unreadable pages

    full_text = "\n".join(pages_text)
    cleaned   = clean_pdf_text(full_text)

    return {
        "filename": pdf_path.name,
        "title":    str(meta.get('/Title', pdf_path.stem)),
        "author":   str(meta.get('/Author', 'Unknown')),
        "pages":    len(reader.pages),
        "text":     cleaned,
        "char_count": len(cleaned)
    }


print("📄 Extracting text from PDFs...")
documents = []
for pdf_path in tqdm(pdf_files, desc="Extracting"):
    try:
        doc = extract_pdf(pdf_path)
        documents.append(doc)
        print(f"   ✅ {doc['filename']}: {doc['pages']} pages, {doc['char_count']:,} chars")
    except Exception as e:
        print(f"   ❌ Failed: {pdf_path.name} — {e}")

print(f"\n✅ Extracted {len(documents)} documents")

---
## ✂️ Step 3: Semantic Chunking

We can't embed entire papers — embedding models have token limits (~512 tokens), and large chunks reduce retrieval precision. We use a **sliding window chunker** with:

- **Chunk size:** ~400 words (captures a full argument/finding)
- **Overlap:** 50 words (ensures no finding gets cut off at a boundary)
- **Paragraph-aware:** breaks prefer paragraph boundaries over mid-sentence

Each chunk retains metadata: source paper, page estimate, and position — so we can cite the exact location.

In [ ]:
def chunk_text(
    text: str,
    chunk_size: int = 400,
    overlap: int = 50
) -> List[str]:
    """
    Split text into overlapping word-based chunks, preferring paragraph boundaries.
    """
    # Split into paragraphs first
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

    chunks, current_words, current_chunk = [], 0, []

    for para in paragraphs:
        para_words = para.split()
        if current_words + len(para_words) > chunk_size and current_chunk:
            # Save current chunk
            chunks.append(' '.join(current_chunk))
            # Overlap: carry forward last N words
            overlap_words = current_chunk[-overlap:] if overlap < len(current_chunk) else current_chunk
            current_chunk = overlap_words + para_words
            current_words = len(current_chunk)
        else:
            current_chunk.extend(para_words)
            current_words += len(para_words)

    if current_chunk:
        chunks.append(' '.join(current_chunk))

    return [c for c in chunks if len(c.strip()) > 50]  # Filter near-empty chunks


def build_chunks(documents: List[Dict]) -> List[Dict]:
    """Chunk all documents and build a flat list with metadata."""
    all_chunks = []
    for doc in documents:
        chunks = chunk_text(doc['text'])
        for i, chunk_text_str in enumerate(chunks):
            # Deterministic ID from content hash
            chunk_id = hashlib.md5(f"{doc['filename']}_{i}".encode()).hexdigest()[:16]
            all_chunks.append({
                "id":       chunk_id,
                "text":     chunk_text_str,
                "filename": doc['filename'],
                "title":    doc['title'],
                "author":   doc['author'],
                "chunk_idx": i,
                "total_chunks": len(chunks)
            })
    return all_chunks


all_chunks = build_chunks(documents)

print(f"✅ Chunking complete")
print(f"   Documents: {len(documents)}")
print(f"   Total chunks: {len(all_chunks):,}")
if documents:
    avg_chunks = len(all_chunks) / len(documents)
    print(f"   Avg chunks per paper: {avg_chunks:.1f}")
    # Show sample chunk
    sample = all_chunks[0]
    print(f"\n📝 Sample chunk from '{sample['filename']}':\n")
    print(sample['text'][:300] + "...")

---
## 🔢 Step 4: Generate Biomedical Embeddings

We use **`pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb`** — a BioBERT model fine-tuned specifically for semantic similarity in biomedical text. It outperforms generic sentence transformers on tasks like:

- Matching a query like *"SHANK3 methylation in autism"* to a passage that uses *"SHANK3 CpG island hypermethylation in ASD"*
- Understanding that *"variant of uncertain significance"* and *"VUS"* mean the same thing

Each chunk becomes a 768-dimensional vector. Semantically similar chunks cluster nearby in this space — enabling fast nearest-neighbor retrieval.

In [ ]:
# Load the biomedical embedding model
MODEL_NAME = "pritamdeka/BioBERT-mnli-snli-scinli-scitail-mednli-stsb"
print(f"🤖 Loading embedding model: {MODEL_NAME}")
print("   (First run downloads ~400MB — cached for future runs)")

embedder = SentenceTransformer(MODEL_NAME)
print(f"✅ Model loaded | Embedding dim: {embedder.get_sentence_embedding_dimension()}")

In [ ]:
def embed_chunks(chunks: List[Dict], embedder, batch_size: int = 32) -> np.ndarray:
    """Embed all chunks in batches with progress tracking."""
    texts = [c['text'] for c in chunks]
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        embeddings = embedder.encode(
            batch,
            convert_to_numpy=True,
            normalize_embeddings=True,  # Cosine similarity optimization
            show_progress_bar=False
        )
        all_embeddings.append(embeddings)

    return np.vstack(all_embeddings) if all_embeddings else np.array([])


if all_chunks:
    print(f"🔢 Embedding {len(all_chunks):,} chunks...")
    embeddings = embed_chunks(all_chunks, embedder)
    print(f"✅ Embeddings shape: {embeddings.shape}")
else:
    print("⚠️  No chunks to embed — upload PDFs first!")
    embeddings = np.array([])

---
## 🗄️ Step 5: Store in ChromaDB

**ChromaDB** is a vector database that runs entirely in-process — no server, no configuration, no cost. It stores:
- The embedding vectors (for fast similarity search)
- The original text (returned with results)
- Metadata (source paper, author, chunk position) for citation

The database persists to disk, so you don't need to re-embed every session — just load it.

In [ ]:
# Initialize ChromaDB with persistent storage
chroma_client = chromadb.PersistentClient(path=str(DB_DIR))

# Create or load collection
COLLECTION_NAME = "biorag_papers"
try:
    collection = chroma_client.get_collection(COLLECTION_NAME)
    print(f"📂 Loaded existing collection: '{COLLECTION_NAME}' ({collection.count()} chunks)")
except Exception:
    collection = chroma_client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"}  # Use cosine similarity
    )
    print(f"✨ Created new collection: '{COLLECTION_NAME}'")

In [ ]:
def insert_chunks_to_chroma(chunks, embeddings, collection, batch_size=100):
    """Insert chunks and embeddings into ChromaDB in batches."""
    if len(chunks) == 0:
        print("⚠️  No chunks to insert")
        return

    # Get existing IDs to avoid duplicates
    existing = set(collection.get(include=[])['ids'])
    new_chunks     = [c for c in chunks if c['id'] not in existing]
    new_embeddings = embeddings[[i for i,c in enumerate(chunks) if c['id'] not in existing]]

    if len(new_chunks) == 0:
        print("✅ All chunks already in database (skipping)")
        return

    print(f"➕ Inserting {len(new_chunks):,} new chunks...")
    for i in tqdm(range(0, len(new_chunks), batch_size), desc="Inserting"):
        batch_chunks     = new_chunks[i : i+batch_size]
        batch_embeddings = new_embeddings[i : i+batch_size]

        collection.add(
            ids        = [c['id'] for c in batch_chunks],
            embeddings = batch_embeddings.tolist(),
            documents  = [c['text'] for c in batch_chunks],
            metadatas  = [{
                'filename':   c['filename'],
                'title':      c['title'][:200],
                'author':     c['author'][:100],
                'chunk_idx':  c['chunk_idx'],
            } for c in batch_chunks]
        )

    print(f"✅ Collection now has {collection.count():,} total chunks")


if len(all_chunks) > 0 and len(embeddings) > 0:
    insert_chunks_to_chroma(all_chunks, embeddings, collection)
else:
    print("⚠️  Upload PDFs first, then re-run from Step 1")

---
## 🔍 Step 6: Test Retrieval

Before adding the LLM layer, let's verify that semantic search is working. Good retrieval is the foundation — if the wrong passages are retrieved, even the best LLM can't give a good answer.

Modify the test queries below to match your actual papers.

In [ ]:
def retrieve(query: str, collection, embedder, top_k: int = 5) -> List[Dict]:
    """
    Retrieve the top-k most semantically relevant chunks for a query.
    Returns list of dicts with text, metadata, and similarity score.
    """
    query_embedding = embedder.encode(
        [query],
        normalize_embeddings=True,
        convert_to_numpy=True
    )

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=min(top_k, collection.count()),
        include=['documents', 'metadatas', 'distances']
    )

    output = []
    for doc, meta, dist in zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ):
        output.append({
            'text':       doc,
            'filename':   meta.get('filename', ''),
            'title':      meta.get('title', ''),
            'author':     meta.get('author', ''),
            'chunk_idx':  meta.get('chunk_idx', 0),
            'similarity': round(1 - dist, 4)  # Convert distance to similarity
        })
    return output


# ✏️ Modify these queries to match your papers!
TEST_QUERIES = [
    "What methylation QC thresholds are used for IDAT array data?",
    "How do epilepsy-associated variants affect gene expression?",
    "What is the role of DNA methylation in pediatric brain tumors?",
]

if collection.count() > 0:
    for query in TEST_QUERIES:
        print(f"\n{'='*60}")
        print(f"🔍 Query: {query}")
        print('='*60)
        results = retrieve(query, collection, embedder, top_k=3)
        for i, r in enumerate(results, 1):
            print(f"\n  [{i}] 📄 {r['filename']} (similarity: {r['similarity']:.3f})")
            print(f"      {r['text'][:250]}...")
else:
    print("⚠️  Database is empty — upload PDFs and run ingestion first")

---
## 💾 Step 7: Save Database + Index Manifest

The ChromaDB database is already persisted to `./biorag_db/`. We also save a human-readable manifest of all indexed papers — useful for documentation and for passing to Notebook 4.

In [ ]:
# Save index manifest
manifest = []
for doc in documents:
    chunks_for_doc = [c for c in all_chunks if c['filename'] == doc['filename']]
    manifest.append({
        'filename':     doc['filename'],
        'title':        doc['title'],
        'author':       doc['author'],
        'pages':        doc['pages'],
        'char_count':   doc['char_count'],
        'num_chunks':   len(chunks_for_doc)
    })

manifest_path = DB_DIR / "index_manifest.json"
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2)

print("✅ BioRAG index saved!")
print(f"   Vector DB:  {DB_DIR}/")
print(f"   Manifest:   {manifest_path}")
print(f"   Papers indexed: {len(documents)}")
print(f"   Total chunks:   {collection.count():,}")
print("\n📋 Indexed papers:")
for m in manifest:
    print(f"   • {m['filename']} — {m['num_chunks']} chunks")

print("\n🚀 Ready for Notebook 4: BioRAG Q&A with Claude!")

---
## ✅ Summary

In this notebook we:

1. **Uploaded PDFs** — directly in Colab, with privacy preserved
2. **Extracted and cleaned text** — removed noise, fixed PDF artifacts, stripped reference sections
3. **Chunked documents** — sliding window with paragraph-aware boundaries and overlap
4. **Generated biomedical embeddings** — using BioBERT fine-tuned for semantic similarity
5. **Stored in ChromaDB** — persistent vector database with full metadata for citation
6. **Validated retrieval** — confirmed that semantic search finds relevant passages

**➡️ Next: [Notebook 4 — BioRAG Q&A with Claude](./04_BioRAG_QA_Demo.ipynb)**

In Notebook 4, we connect this retrieval system to Claude via the Anthropic API to build a full conversational Q&A system that cites exact passages from your papers.

---
*BioRAG is an open-source research tool. It is not a substitute for reading the original papers — always verify AI-synthesized answers against source documents.*